# Cohere Transcribe Lab

This notebook showcases the [`cohere-transcribe-03-2026`](https://docs.cohere.com/v2/changelog/cohere-transcribe-03-2026) model — Cohere's first automatic speech recognition (ASR) model. It is a 2B-parameter Conformer-based model that supports 14 languages: English, German, French, Italian, Spanish, Portuguese, Greek, Dutch, Polish, Vietnamese, Chinese, Arabic, Japanese, and Korean.

**What we'll cover:**
1. Transcribing short audio with background noise
2. Near real-time transcription via chunked API calls
3. Multilingual transcription across all 14 supported languages
4. *(Optional)* Real-time streaming using the model running locally — requires GPU for best performance

---
**Setup:** Copy `.env.example` to `.env` and add your Cohere API key before running cells 1–3.


In [ ]:
# Install dependencies (run once)
!pip install -r requirements.txt

In [ ]:
import os
import io
import time
import requests
import numpy as np
import soundfile as sf
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Audio, display, Markdown
import cohere

load_dotenv()

api_key = os.getenv("CO_API_KEY")
if not api_key:
    raise EnvironmentError("CO_API_KEY not found. Copy .env.example to .env and add your key.")

co = cohere.ClientV2(api_key=api_key)

AUDIO_DIR = Path("audio")
AUDIO_DIR.mkdir(exist_ok=True)
(AUDIO_DIR / "multilingual").mkdir(exist_ok=True)

MODEL = "cohere-transcribe-03-2026"
HF_BASE = "https://huggingface.co/spaces/CohereLabs/cohere-transcribe-03-2026/resolve/main/examples/audio"

def download_audio(lang: str, dest: Path) -> Path:
    """Download a FLEURS audio sample from the Cohere HuggingFace Space."""
    if dest.exists():
        return dest
    url = f"{HF_BASE}/{lang}.wav"
    print(f"Downloading {lang}.wav ...")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    dest.write_bytes(r.content)
    print(f"  Saved to {dest}")
    return dest

print("Setup complete. Cohere client initialised.")

---
## 1: Transcribing Short Audio with Background Noise

Real-world audio rarely arrives clean. This cell demonstrates that Cohere Transcribe is robust to background noise.

We take the FLEURS English validation clip, synthetically add cafeteria-style Gaussian noise (SNR ≈ 10 dB), and transcribe the result. Both the clean and noisy audio players are rendered so you can hear the difference yourself.

In [ ]:
# ── Download clean English FLEURS sample ────────────────────────────────────
clean_path = download_audio("en", AUDIO_DIR / "en_clean.wav")

# ── Add background noise ─────────────────────────────────────────────────────
audio_data, sample_rate = sf.read(clean_path)

# Target SNR in dB (lower = noisier)
target_snr_db = 10

signal_power = np.mean(audio_data ** 2)
noise_power_target = signal_power / (10 ** (target_snr_db / 10))
noise = np.random.default_rng(42).normal(0, np.sqrt(noise_power_target), audio_data.shape)

noisy_audio = np.clip(audio_data + noise, -1.0, 1.0)
noisy_path = AUDIO_DIR / "en_noisy.wav"
sf.write(noisy_path, noisy_audio, sample_rate)
print(f"Noisy audio saved → {noisy_path}  (SNR ≈ {target_snr_db} dB)")

# ── Listen to both ───────────────────────────────────────────────────────────
display(Markdown("**Clean audio:**"))
display(Audio(str(clean_path)))
display(Markdown("**Noisy audio (SNR ≈ 10 dB):**"))
display(Audio(str(noisy_path)))

# ── Transcribe the noisy file ────────────────────────────────────────────────
print("\nTranscribing noisy audio...")
t0 = time.time()

with open(noisy_path, "rb") as f:
    response = co.audio.transcriptions.create(
        model=MODEL,
        language="en",
        file=f,
    )

elapsed = time.time() - t0
print(f"\nTranscription ({elapsed:.2f}s):")
print(f"  {response.text}")

---
## 2: Near Real-Time Transcription

The Cohere Transcribe API is file-based, but near real-time transcription is achievable by streaming microphone audio in short chunks and sending each chunk to the API as it is captured.

- **Primary path:** records from your microphone in 4-second windows. Each window is immediately sent to the API and the transcription is printed with a timestamp.
- **Fallback path (no mic):** chunks the clean English WAV file into 4-second segments and processes them sequentially to simulate the same effect.

Set `USE_MIC = True` to use your microphone, or `False` to use the fallback.

In [ ]:
import threading
import queue

USE_MIC = False          # Set True to record from your microphone
CHUNK_SECONDS = 4        # Length of each audio chunk
RECORD_SECONDS = 20      # Total mic recording duration (mic mode only)
SAMPLE_RATE = 16_000     # 16 kHz mono — optimal for ASR


def transcribe_chunk(chunk: np.ndarray, sr: int, chunk_idx: int) -> None:
    """Write chunk to an in-memory WAV buffer and transcribe it."""
    buf = io.BytesIO()
    sf.write(buf, chunk, sr, format="WAV", subtype="PCM_16")
    buf.seek(0)
    buf.name = "chunk.wav"  # cohere SDK needs a name attribute

    try:
        result = co.audio.transcriptions.create(
            model=MODEL,
            language="en",
            file=buf,
        )
        ts = time.strftime("%H:%M:%S")
        print(f"[{ts}] Chunk {chunk_idx:02d}: {result.text}")
    except Exception as exc:
        print(f"[Chunk {chunk_idx:02d}] Error: {exc}")


if USE_MIC:
    try:
        import sounddevice as sd
    except ImportError:
        raise ImportError("Install sounddevice: pip install sounddevice")

    chunk_samples = CHUNK_SECONDS * SAMPLE_RATE
    audio_queue: queue.Queue = queue.Queue()

    def mic_callback(indata, frames, time_info, status):
        audio_queue.put(indata.copy())

    print(f"Recording for {RECORD_SECONDS}s in {CHUNK_SECONDS}s chunks. Speak now...\n")

    buffer = []
    chunk_idx = 0
    start = time.time()

    with sd.InputStream(samplerate=SAMPLE_RATE, channels=1, dtype="float32",
                        blocksize=CHUNK_SECONDS * SAMPLE_RATE,
                        callback=mic_callback):
        while time.time() - start < RECORD_SECONDS:
            try:
                chunk_data = audio_queue.get(timeout=CHUNK_SECONDS + 1)
                chunk_idx += 1
                t = threading.Thread(target=transcribe_chunk,
                                     args=(chunk_data.flatten(), SAMPLE_RATE, chunk_idx))
                t.start()
            except queue.Empty:
                break

    print("\nRecording complete.")

else:
    # ── File-based fallback ──────────────────────────────────────────────────
    print("File-based fallback mode: chunking en_clean.wav into 4-second segments.\n")

    audio_data, sr = sf.read(clean_path)
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)  # stereo → mono

    chunk_size = CHUNK_SECONDS * sr
    chunks = [
        audio_data[i : i + chunk_size]
        for i in range(0, len(audio_data), chunk_size)
        if len(audio_data[i : i + chunk_size]) > sr // 2  # skip sub-0.5s tail
    ]

    print(f"Total chunks: {len(chunks)} × {CHUNK_SECONDS}s\n")

    threads = []
    for idx, chunk in enumerate(chunks, start=1):
        t = threading.Thread(target=transcribe_chunk, args=(chunk, sr, idx))
        t.start()
        threads.append(t)
        time.sleep(0.3)  # slight stagger to avoid rate-limit bursts

    for t in threads:
        t.join()

    print("\nDone.")

---
## 3: Multilingual Transcription

Cohere Transcribe supports 14 languages out of the box. This cell downloads all 14 FLEURS validation clips — English, French, German, Spanish, Portuguese, Italian, Dutch, Polish, Greek, Arabic, Japanese, Korean, Chinese, and Vietnamese — from the official Cohere HuggingFace Space and transcribes each one using the correct [ISO-639-1](https://en.wikipedia.org/wiki/List_of_ISO_639_language_codes) language code.

> **Note:** The model is single-language per request. You must specify the correct `language` parameter for best accuracy; it does not perform automatic language detection.

In [ ]:
ALL_LANGUAGES = [
    {"code": "en", "label": "English",    "flag": "🇺🇸"},
    {"code": "fr", "label": "French",     "flag": "🇫🇷"},
    {"code": "de", "label": "German",     "flag": "🇩🇪"},
    {"code": "es", "label": "Spanish",    "flag": "🇪🇸"},
    {"code": "pt", "label": "Portuguese", "flag": "🇧🇷"},
    {"code": "it", "label": "Italian",    "flag": "🇮🇹"},
    {"code": "nl", "label": "Dutch",      "flag": "🇳🇱"},
    {"code": "pl", "label": "Polish",     "flag": "🇵🇱"},
    {"code": "el", "label": "Greek",      "flag": "🇬🇷"},
    {"code": "ar", "label": "Arabic",     "flag": "🇸🇦"},
    {"code": "ja", "label": "Japanese",   "flag": "🇯🇵"},
    {"code": "ko", "label": "Korean",     "flag": "🇰🇷"},
    {"code": "zh", "label": "Chinese",    "flag": "🇨🇳"},
    {"code": "vi", "label": "Vietnamese", "flag": "🇻🇳"},
]

# ── Select languages to transcribe ───────────────────────────────────────────
# By default all 14 languages are included. To run a subset, replace RUN_CODES
# with a list of ISO-639-1 codes, e.g. ["en", "fr", "ja"]
RUN_CODES = [lang["code"] for lang in ALL_LANGUAGES]  # all by default
# RUN_CODES = ["en", "fr", "ja"]                      # example subset

LANGUAGES = [lang for lang in ALL_LANGUAGES if lang["code"] in RUN_CODES]

results = []

for lang in LANGUAGES:
    code = lang["code"]
    path = AUDIO_DIR / "multilingual" / f"{code}.wav"

    print(f"{lang['flag']}  Transcribing {lang['label']} ({code})...")
    t0 = time.time()

    with open(path, "rb") as f:
        resp = co.audio.transcriptions.create(
            model=MODEL,
            language=code,
            file=f,
        )

    elapsed = time.time() - t0
    results.append({
        "lang": lang,
        "text": resp.text,
        "elapsed": elapsed,
        "path": path,
    })
    print(f"   → {resp.text}  ({elapsed:.2f}s)\n")

# ── Summary table ────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print(f"{'Language':<12} {'Code':<6} {'Time':>6}  Transcription")
print("=" * 70)
for r in results:
    lang = r["lang"]
    snippet = r["text"][:60] + ("..." if len(r["text"]) > 60 else "")
    print(f"{lang['flag']} {lang['label']:<10} {lang['code']:<6} {r['elapsed']:>5.2f}s  {snippet}")
print("=" * 70)

# ── Audio players ────────────────────────────────────────────────────────────
print("\nAudio players:")
for r in results:
    lang = r["lang"]
    display(Markdown(f"**{lang['flag']} {lang['label']}:** {r['text']}"))
    display(Audio(str(r["path"])))

---
## 4: OPTIONAL Near Real-Time Transcoding with Transcribe Model running locally

This cell runs the Cohere Transcribe model **entirely on your own machine** using the HuggingFace `transformers` library. Because inference is local, there are no Cohere API keys, no rate limits, and no file-size restrictions — audio is captured from your microphone in configurable chunks and transcribed as each chunk arrives.

### Prerequisites

| Requirement | Notes |
| --- | --- |
| **HuggingFace account** | Visit [CohereLabs/cohere-transcribe-03-2026](https://huggingface.co/CohereLabs/cohere-transcribe-03-2026) and accept the model license |
| **HF_TOKEN** | Generate a token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and add it to your `.env` file |
| **Model weights** | ~4 GB downloaded from HuggingFace on first run (cached locally afterward) |
| **GPU (recommended)** | NVIDIA CUDA gives ~525× real-time factor. Apple M-series MPS also works. CPU is functional but noticeably slower. |
| **Extra dependencies** | `torch`, `transformers` — already in `requirements.txt` |

> **Tip:** Adjust `CHUNK_SECONDS` to trade latency for accuracy. Shorter chunks (2–3 s) feel more real-time; longer chunks (5–10 s) give the model more context and produce better transcriptions. Values below 1 s are too short for meaningful ASR output.


In [ ]:
# ── Install dependencies (run once, then restart kernel) ────────────────────
# !pip install -r requirements.txt

import os
import numpy as np
import sounddevice as sd
import time
import torch
from dotenv import load_dotenv
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

load_dotenv()

# ── HuggingFace authentication ───────────────────────────────────────────────
# The model is gated — accept the license at:
# https://huggingface.co/CohereLabs/cohere-transcribe-03-2026
# Then add HF_TOKEN to your .env file (see README for steps)
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise EnvironmentError(
        "HF_TOKEN not found. Add it to your .env file.\n"
        "Steps: https://github.com/andytran24/transcribe-lab#optional-cell-4-setup-local-model-streaming"
    )

# ── Configuration ────────────────────────────────────────────────────────────
MODEL_ID       = "CohereLabs/cohere-transcribe-03-2026"
LANGUAGE       = "en"    # ISO-639-1 code for the language being spoken
CHUNK_SECONDS  = 4       # seconds per chunk (2–10 recommended; shorter = lower latency)
SAMPLE_RATE    = 16_000  # Hz — model expects 16 kHz mono
RECORD_SECONDS = 30      # total recording duration — press Jupyter ■ Stop to end early
USE_MIC        = False   # True = record from mic | False = use audio/en_clean.wav

# ── Device selection ─────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"   # Apple Silicon GPU
else:
    device = "cpu"
    print("Warning: No GPU detected — running on CPU. Expect noticeable lag per chunk.")

print(f"Device: {device}")

# ── Load model (downloads ~4 GB on first run, then cached) ───────────────────
print(f"Loading {MODEL_ID} ...")
processor = AutoProcessor.from_pretrained(
    MODEL_ID, token=hf_token, trust_remote_code=True
)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    token=hf_token,
    trust_remote_code=True,
    dtype=torch.float16 if device != "cpu" else torch.float32,
).to(device)
model.eval()
print("Model ready.\n")


def transcribe_chunk_local(audio_np: np.ndarray, chunk_idx: int) -> None:
    """Transcribe one audio chunk using the model's built-in transcribe() method."""
    t0 = time.time()
    # The model exposes a high-level transcribe() that handles feature extraction,
    # chunking, generation, and decoding internally.
    texts = model.transcribe(
        processor,
        language=LANGUAGE,
        audio_arrays=[audio_np.astype(np.float32)],
        sample_rates=[SAMPLE_RATE],
    )
    elapsed = time.time() - t0
    transcription = texts[0].strip() if texts else ""
    ts = time.strftime("%H:%M:%S")
    print(f"[{ts}] Chunk {chunk_idx:02d} ({elapsed:.2f}s): {transcription if transcription else '(silence)'}")


# ── Input source ─────────────────────────────────────────────────────────────
if USE_MIC:
    print(f"Recording for up to {RECORD_SECONDS}s in {CHUNK_SECONDS}s chunks.")
    print("Speak into your microphone. Press the Jupyter \u25a0 Stop button to end early.\n")
    chunk_samples = CHUNK_SECONDS * SAMPLE_RATE
    chunk_idx = 0
    start = time.time()

    try:
        with sd.InputStream(samplerate=SAMPLE_RATE, channels=1, dtype="float32") as stream:
            while time.time() - start < RECORD_SECONDS:
                remaining = RECORD_SECONDS - (time.time() - start)
                print(f"  Recording chunk {chunk_idx + 1} ({min(CHUNK_SECONDS, remaining):.0f}s)...", end="", flush=True)
                audio_chunk, _ = stream.read(chunk_samples)
                chunk_idx += 1
                print(" transcribing...", end="", flush=True)
                transcribe_chunk_local(audio_chunk.flatten(), chunk_idx)
    except KeyboardInterrupt:
        print("\nStopped by user.")

    print("\nRecording complete.")

else:
    # ── File-based fallback ───────────────────────────────────────────────────
    import soundfile as sf
    from pathlib import Path

    file_path = Path("audio/en_clean.wav")
    print(f"File mode: chunking {file_path} into {CHUNK_SECONDS}s segments.")
    print("Tip: Set USE_MIC = True above to transcribe live from your microphone.\n")

    audio_data, sr = sf.read(file_path)
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)           # stereo -> mono
    if sr != SAMPLE_RATE:
        import scipy.signal as sps
        samples = int(len(audio_data) * SAMPLE_RATE / sr)
        audio_data = sps.resample(audio_data, samples).astype(np.float32)

    chunk_size = CHUNK_SECONDS * SAMPLE_RATE
    chunks = [
        audio_data[i : i + chunk_size]
        for i in range(0, len(audio_data), chunk_size)
        if len(audio_data[i : i + chunk_size]) > SAMPLE_RATE // 2
    ]
    print(f"Total chunks: {len(chunks)} x {CHUNK_SECONDS}s\n")

    for idx, chunk in enumerate(chunks, start=1):
        print(f"  Chunk {idx:02d}: transcribing...", end="", flush=True)
        transcribe_chunk_local(chunk.astype(np.float32), idx)

    print("\nDone.")